In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# BigQuery Agent Analytics — the four-guarantee decision-lineage demo

**You own the graph. The SDK validates it cheaply, extracts deterministically, and
resolves user inputs to canonical concepts.**

This notebook walks through the four guarantees the SDK ships post-V5:

| Guarantee | What it means | Beat |
|---|---|---|
| **Own** | The user authors `CREATE PROPERTY GRAPH`. The SDK populates base tables — it never rewrites your graph DDL. | 1 |
| **Validate** | A sub-second pre-flight catches binding/table drift before extraction spends a dollar. | 2 |
| **Extract cheaply** | Deterministic compiled extractors handle structured events; `AI.GENERATE` only fills semantic gaps. | 3 |
| **Resolve** | User-typed inputs (`"Consumer Banking"`) resolve to canonical concept names (`skos:RetailBanking`) via the SKOS taxonomy you authored once. | 4 |

The demo's domain is **MAKO** — the Monetization Agents Knowledge Ontology, a real
Yahoo Monetization Platform ontology covering audience-segment / bid-value / creative-
variant / frequency-cap decisions. The agent that produces the events is a real ADK
Gemini agent talking to the BQ AA plugin; the events you see below were not synthesized.

## Section 0 — what you need to bring

**Minimum hand-authored input: one ontology file.**

Everything else — `binding.yaml`, `table_ddl.sql`, the property graph, the trace
events — is either auto-generated by the SDK's CLI, owned by you (your tables, your
graph DDL), or emitted by the BQ AA plugin when an agent runs. Two YAML files is
the *common* shape, but only one is *required*.

Three input shapes are equivalent today:

| Input shape | What you author | What's generated |
|---|---|---|
| **(a) Hand-authored YAML** | `ontology.yaml` | `binding.yaml` (`gm scaffold`), `table_ddl.sql` |
| **(b) OWL/SKOS TTL** | `*.ttl` | `ontology.yaml` (`gm import-owl`), then (a) |
| **(c) Future `@builtin:adk-events`** | nothing | everything |

This notebook uses shape **(b)**: `examples/migration_v5/mako_core.ttl` is the
authored input. The TTL → ontology → binding → DDL pipeline lives in
`mako_artifacts.py` (a thin convenience wrapper around `gm import-owl` + `gm scaffold`
tuned for this demo's six-entity scope).

### Install + authenticate + configure

Install the SDK, the ontology package, and the BQ AA plugin. The agent uses Vertex
AI Gemini, so the runtime needs Vertex access on `PROJECT_ID` and BigQuery write
access on the same project.

In [ ]:
!pip install -q bigquery-agent-analytics bigquery-ontology \
    google-adk[vertexai] google-cloud-bigquery pyyaml rdflib python-dotenv


In [ ]:
import os

try:
    from google.colab import auth as _colab_auth
    _colab_auth.authenticate_user()
    print("Colab authentication successful.")
except ImportError:
    print("Not running in Colab — using application-default credentials.")

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "your-project-id")
DATASET_BASE = os.environ.get("BQ_DATASET", "migration_v5_demo")
AGENT_LOCATION = os.environ.get("DEMO_AGENT_LOCATION", "us-central1")
DATASET_LOCATION = os.environ.get("DATASET_LOCATION", "US")

print(f"Project  : {PROJECT_ID}")
print(f"Agent location : {AGENT_LOCATION}")
print(f"Dataset location : {DATASET_LOCATION}")
assert PROJECT_ID != "your-project-id", (
    "Set GOOGLE_CLOUD_PROJECT before running this notebook."
)


### Scratch dataset + feature flags

Every run creates a fresh `migration_v5_demo_<8-hex>` dataset with 1-hour table TTL,
so re-running the notebook never collides with a previous run.

`FEATURES` gates each guarantee's cells. Flip an entry to `False` if the underlying
issue hasn't shipped in your installed SDK — the gated cells degrade to
`Skipped: requires #N` markdown rather than failing.

In [ ]:
import uuid

from google.cloud import bigquery

RUN_ID = uuid.uuid4().hex[:8]
DATASET_ID = f"{DATASET_BASE}_{RUN_ID}"

bq = bigquery.Client(project=PROJECT_ID, location=DATASET_LOCATION)
ds = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
ds.location = DATASET_LOCATION
ds.default_table_expiration_ms = 3600_000  # 1 hour
bq.create_dataset(ds, exists_ok=True)
print(f"Scratch dataset: {PROJECT_ID}.{DATASET_ID}  (TTL 1h, location {DATASET_LOCATION})")

FEATURES = {
    "skip_property_graph": True,    # #104
    "binding_validate": True,       # #105
    "validate_extracted_graph": True,  # #76
    "compiled_extractors_c1": True,  # #75 PR 4b/4c — compile + measurement
    "compiled_extractors_c2": True,  # #75 C2 — runtime bundle loading
    "concept_index_reader": True,    # #58 reader follow-on
}
print("FEATURES:")
for k, v in FEATURES.items():
    print(f"  {k:30s} {'ON' if v else 'off'}")


### Generate ontology + binding + DDL from the MAKO TTL

`mako_artifacts.regenerate_snapshots(project, dataset)` runs the equivalent of

```
gm import-owl mako_core.ttl --out ontology.yaml
gm scaffold --ontology ontology.yaml --project P --dataset D --out .
```

plus the demo-specific post-processing this notebook needs: drops cross-namespace
PROV-O / PKO relationships (the OWL importer can't resolve those endpoints), restricts
the binding to the six demo entities (`AgentSession`, `DecisionExecution`,
`DecisionPoint`, `Candidate`, `SelectionOutcome`, `ContextSnapshot`), and maps each
ontology property type to its BQ column type. The output is four files:

- `ontology.yaml` — 18 MAKO entities with primary keys resolved
- `binding.yaml` — 6 entities + 9 relationships against `{PROJECT_ID}.{DATASET_ID}`
- `table_ddl.sql` — `CREATE TABLE IF NOT EXISTS` for every node + edge table, including `session_id STRING, extracted_at TIMESTAMP` SDK metadata columns
- `property_graph.sql` — `CREATE OR REPLACE PROPERTY GRAPH` over the same tables

In [ ]:
import sys
sys.path.insert(0, "examples/migration_v5")

import mako_artifacts

artifact_counts = mako_artifacts.regenerate_snapshots(
    project=PROJECT_ID,
    dataset=DATASET_ID,
)
print(artifact_counts)

from pathlib import Path
HERE = Path("examples/migration_v5")
ONTOLOGY_PATH = HERE / "ontology.yaml"
BINDING_PATH = HERE / "binding.yaml"
TABLE_DDL_PATH = HERE / "table_ddl.sql"
PROPERTY_GRAPH_PATH = HERE / "property_graph.sql"
print()
for p in (ONTOLOGY_PATH, BINDING_PATH, TABLE_DDL_PATH, PROPERTY_GRAPH_PATH):
    print(f"{p}  {p.stat().st_size:>6} bytes")


### Apply the table DDL

Run the `CREATE TABLE IF NOT EXISTS` block — fifteen statements, one per node + edge
table. The plugin's own `agent_events` table is created separately when the plugin
starts; this DDL just creates the MAKO graph tables that the SDK will materialize
into.

In [ ]:
ddl = TABLE_DDL_PATH.read_text()
for stmt in ddl.split(";"):
    stmt = stmt.strip()
    if not stmt:
        continue
    bq.query(stmt + ";").result()
print(f"Applied {ddl.count(';')} DDL statements against {DATASET_ID}.")


### Run the MAKO agent — populate `agent_events`

`run_agent.py` drives a real ADK Gemini agent through `N` MAKO decision sessions.
Each session walks the canonical decision flow:

```
capture_context  →  propose_decision_point  →  evaluate_candidate (×3-5)  →  commit_outcome  →  complete_execution
```

The BQ AA plugin attached to the runner captures every invocation, agent, LLM, tool,
and HITL event into `{PROJECT_ID}.{DATASET_ID}.agent_events` as a real plugin trace.
Nothing is synthesized.

**This cell is the only one that costs Vertex tokens.** Subsequent cells read from the
table we just populated. Keep `--sessions` small (10-50) while iterating.

In [ ]:
import subprocess

SESSIONS = int(os.environ.get("MIGRATION_V5_SESSIONS", "10"))
cmd = [
    sys.executable,
    "examples/migration_v5/run_agent.py",
    "--sessions", str(SESSIONS),
    "--project", PROJECT_ID,
    "--dataset", DATASET_ID,
    "--location", DATASET_LOCATION,
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# Sanity check: count rows by event_type in agent_events.
rows = bq.query(f"""
    SELECT event_type, COUNT(*) AS n
    FROM `{PROJECT_ID}.{DATASET_ID}.agent_events`
    GROUP BY event_type
    ORDER BY n DESC
""").result()
for r in rows:
    print(f"  {r.event_type:30s} {r.n}")


Section 0 is done. We have:

- A scratch dataset with the MAKO node + edge tables created (empty).
- An `agent_events` table populated by a real ADK Gemini agent + BQ AA plugin.
- Authored `ontology.yaml` + `binding.yaml` + `property_graph.sql`.

The four-guarantee story starts in Section 1.

## Section 1 — Beat 1: you own the graph definition (#104)

**Guarantee:** Own.

_To be filled in pass 2._

## Section 2 — Beat 2: pre-flight catches binding drift before you spend a dollar (#105)

**Guarantee:** Validate.

_To be filled in pass 3._

## Section 3 — Beat 3: structured events extract deterministically; LLM only fills the gaps (#75 + #76)

**Guarantee:** Extract cheaply.

_To be filled in pass 4._

## Section 4 — Beat 4: user-typed inputs resolve to canonical concepts (#58)

**Guarantee:** Resolve.

_To be filled in pass 5._

## Section 5 — close

### Four-guarantee recap

| Guarantee | Before | After |
|---|---|---|
| **Own** | `CREATE OR REPLACE PROPERTY GRAPH` every build (SDK overwrites your DDL) | `--skip-property-graph`; you own the graph object, the SDK populates base tables. |
| **Validate** | First failure is at extraction time, after `AI.GENERATE` already ran | Sub-second pre-flight against live BigQuery schema before extraction starts. |
| **Extract cheaply** | `AI.GENERATE` over the full transcript every session | Compiled deterministic extractors handle structured events; AI fallback only for semantic gaps. Per-session token table shows the savings. |
| **Resolve** | User types `"Consumer Banking"`; GQL returns nothing because the canonical label is `skos:RetailBanking` | SKOS concept-index lookup resolves user-typed inputs to canonical names before GQL runs. |

### Read more

Each guarantee has its own issue + landing PRs:

- **Own** — [#104](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/104)
- **Validate** — [#105](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/105) (pre-flight) + [#76](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/76) (post-extract `ValidationReport`)
- **Extract cheaply** — [#75](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/75) (compile-extractors C1 + C2)
- **Resolve** — [#58](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/58) (concept-index reader; emission shipped in [PR #92](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/pull/92))
- **Storyboard** — [#107](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/107) (this notebook's cell-by-cell plan)

### What's next

- A default `@builtin:adk-events` ontology so users with no domain-specific extraction needs can run the SDK with **zero authored YAML**.
- Phase 2 of [#75](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/75): session-aggregated compilation that reduces per-session AI cost further.
- Additional resolver layers in [#58](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/58) beyond `LabelSynonymResolver` (semantic similarity, embedding-based).